In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

RAW_PATH = os.path.join(BASE_PATH, "1st Raw_Dataset")
PREPROCESS_PATH = os.path.join(BASE_PATH, "3rd Preprocessing")

print("Raw Dataset:", RAW_PATH)
print("Preprocessing:", PREPROCESS_PATH)

Raw Dataset: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/1st Raw_Dataset
Preprocessing: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing


In [3]:
for root, dirs, files in os.walk(RAW_PATH):
    level = root.replace(RAW_PATH, '').count(os.sep)
    indent = '  ' * level
    print(indent + os.path.basename(root) + "/")

1st Raw_Dataset/
  Potato/
    Healthy/
    Early_Blight/
    Late_Blight/


In [4]:
#IMP LIBRARY
import os
import numpy as np
from PIL import Image

#EXTRA LIBRARY
import shutil
import random
import numpy as np
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt


print("Libraries loaded successfully.")

Libraries loaded successfully.


In [5]:
DATASET_PATH = os.path.join(RAW_PATH, "Potato")

print("Dataset path:", DATASET_PATH)
print("Classes:", os.listdir(DATASET_PATH))

Dataset path: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/1st Raw_Dataset/Potato
Classes: ['Healthy', 'Early_Blight', 'Late_Blight']


In [6]:
images = []
labels = []

classes = ["Healthy", "Early_Blight", "Late_Blight"]

for label, class_name in enumerate(classes):

    class_path = os.path.join(DATASET_PATH, class_name)

    for file_name in os.listdir(class_path):

        file_path = os.path.join(class_path, file_name)

        try:
            image = Image.open(file_path).convert("RGB")

            images.append(np.array(image))
            labels.append(label)

        except Exception as e:
            print("Error:", file_path, e)

print("Total images loaded:", len(images))
print("Total labels:", len(labels))

Total images loaded: 2152
Total labels: 2152


In [7]:
from collections import Counter

label_counts = Counter(labels)

print("Class distribution:")

for label, count in label_counts.items():
    print(classes[label], ":", count)

Class distribution:
Healthy : 152
Early_Blight : 1000
Late_Blight : 1000


In [8]:
#Resize + Normalize

IMG_SIZE = (224, 224)

X = []

for image in images:
    image = Image.fromarray(image)
    image = image.resize(IMG_SIZE)

    image = np.array(image) / 255.0

    X.append(image)

X = np.array(X, dtype=np.float32)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Pixel minimum:", X.min())
print("Pixel maximum:", X.max())

X shape: (2152, 224, 224, 3)
y shape: (2152,)
Pixel minimum: 0.0
Pixel maximum: 1.0


In [9]:
#Stratified Train / Validation / Test Split

from sklearn.model_selection import train_test_split

# First split: 80% training, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: temporary 20% into 10% validation + 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (1721, 224, 224, 3) (1721,)
Validation: (215, 224, 224, 3) (215,)
Test: (216, 224, 224, 3) (216,)


In [10]:
#augmentation pipeline

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True
)

print("✅ Training augmentation pipeline created.")

✅ Training augmentation pipeline created.


In [11]:
#Verify Distributions
#अब हम confirm करेंगे कि Train / Validation / Test में तीनों classes properly represented हैं

from collections import Counter

def show_distribution(name, labels):
    counts = Counter(labels)

    print(f"\n{name} Distribution:")
    for label, count in sorted(counts.items()):
        print(f"{classes[label]}: {count}")

show_distribution("Training", y_train)
show_distribution("Validation", y_val)
show_distribution("Testing", y_test)


Training Distribution:
Healthy: 121
Early_Blight: 800
Late_Blight: 800

Validation Distribution:
Healthy: 15
Early_Blight: 100
Late_Blight: 100

Testing Distribution:
Healthy: 16
Early_Blight: 100
Late_Blight: 100


In [12]:
#Final Preprocessing Summary
print("========== PREPROCESSING SUMMARY ==========")

print("Original Images:", len(X))
print("Image Size: 224 x 224")
print("Channels: 3 (RGB)")
print("Normalization: 0-1")

print("\nDataset Split:")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print("\nClass Distribution:")
for label, cls in enumerate(classes):
    print(
        f"{cls}:",
        sum(y == label)
    )

print("\nAugmentation:")
print("Training augmentation: Enabled")
print("Validation augmentation: Disabled")
print("Testing augmentation: Disabled")

print("\n✅ PREPROCESSING COMPLETED SUCCESSFULLY")

========== PREPROCESSING SUMMARY ==========
Original Images: 2152
Image Size: 224 x 224
Channels: 3 (RGB)
Normalization: 0-1

Dataset Split:
Training: 1721
Validation: 215
Testing: 216

Class Distribution:
Healthy: 152
Early_Blight: 1000
Late_Blight: 1000

Augmentation:
Training augmentation: Enabled
Validation augmentation: Disabled
Testing augmentation: Disabled

✅ PREPROCESSING COMPLETED SUCCESSFULLY


In [13]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1721, 224, 224, 3)
y_train: (1721,)
X_val: (215, 224, 224, 3)
y_val: (215,)
X_test: (216, 224, 224, 3)
y_test: (216,)


In [14]:
BASE_PATH = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

print(os.listdir(BASE_PATH))


['1st Raw_Dataset', '2nd Data_Verification', '3rd Preprocessing', '4th Model_Training', '5th Model_Evaluation', '6th Trained_Model', '7th Test_Images', '8th Web_Application', '9th Documentation', '10th Presentation PPT']


In [15]:
PREPROCESS_PATH = os.path.join(BASE_PATH, "3rd Preprocessing")

print("Exists:", os.path.exists(PREPROCESS_PATH))

Exists: True


In [16]:
SAVE_PATH = os.path.join(
    PREPROCESS_PATH,
    "processed_potato_data.npz"
)

np.savez(
    SAVE_PATH,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print("✅ Preprocessed dataset saved successfully!")
print("File:", SAVE_PATH)

✅ Preprocessed dataset saved successfully!
File: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/processed_potato_data.npz


In [17]:


file_size_mb = os.path.getsize(SAVE_PATH) / (1024**2)

print("File exists:", os.path.exists(SAVE_PATH))
print(f"File size: {file_size_mb:.2f} MB")

File exists: True
File size: 1235.74 MB


In [18]:
import numpy as np

data = np.load(SAVE_PATH)

print("✅ File loaded successfully!\n")

print("X_train:", data["X_train"].shape)
print("y_train:", data["y_train"].shape)

print("X_val:", data["X_val"].shape)
print("y_val:", data["y_val"].shape)

print("X_test:", data["X_test"].shape)
print("y_test:", data["y_test"].shape)

✅ File loaded successfully!

X_train: (1721, 224, 224, 3)
y_train: (1721,)
X_val: (215, 224, 224, 3)
y_val: (215,)
X_test: (216, 224, 224, 3)
y_test: (216,)


In [19]:
#FINAL PREPROCESSING & DATA SAVING CHECK
import os
import numpy as np

print("=" * 60)
print("       FINAL PREPROCESSING & DATA SAVING CHECK")
print("=" * 60)

# --------------------------------------------------
# 1. File path
# --------------------------------------------------
PROCESSED_FILE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/processed_potato_data.npz"

print("\n📁 Saved File Check")
print("Path:", PROCESSED_FILE)
print("File exists:", os.path.exists(PROCESSED_FILE))

if os.path.exists(PROCESSED_FILE):
    size_gb = os.path.getsize(PROCESSED_FILE) / (1024**3)
    print(f"File size: {size_gb:.2f} GB")

# --------------------------------------------------
# 2. Load processed dataset
# --------------------------------------------------
data = np.load(PROCESSED_FILE)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

# --------------------------------------------------
# 3. Shapes
# --------------------------------------------------
print("\n📊 Dataset Shapes")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

# --------------------------------------------------
# 4. Total images
# --------------------------------------------------
total = len(X_train) + len(X_val) + len(X_test)

print("\n📦 Total Images")
print("Training   :", len(X_train))
print("Validation :", len(X_val))
print("Testing    :", len(X_test))
print("TOTAL      :", total)

# --------------------------------------------------
# 5. Pixel range
# --------------------------------------------------
print("\n🎨 Pixel Range")
print("Minimum pixel value:", X_train.min())
print("Maximum pixel value:", X_train.max())

# --------------------------------------------------
# 6. Class distribution
# --------------------------------------------------
class_names = ["Healthy", "Early_Blight", "Late_Blight"]

print("\n🌱 Class Distribution")

for i, cls in enumerate(class_names):
    print(
        f"{cls:15s} | "
        f"Train: {(y_train == i).sum():4d} | "
        f"Val: {(y_val == i).sum():4d} | "
        f"Test: {(y_test == i).sum():4d}"
    )

# --------------------------------------------------
# 7. Final validation
# --------------------------------------------------
checks = {
    "File exists": os.path.exists(PROCESSED_FILE),
    "Total images = 2152": total == 2152,
    "Image shape correct": X_train.shape[1:] == (224, 224, 3),
    "Train labels match": len(X_train) == len(y_train),
    "Validation labels match": len(X_val) == len(y_val),
    "Test labels match": len(X_test) == len(y_test),
    "Normalization correct": X_train.min() >= 0 and X_train.max() <= 1,
}

print("\n" + "=" * 60)
print("FINAL CHECK RESULTS")
print("=" * 60)

for check, result in checks.items():
    print(("✅" if result else "❌"), check)

if all(checks.values()):
    print("\n🎉 PREPROCESSING + DATA SAVING = 100% VERIFIED")
    print("🚀 Dataset is READY for CNN MODEL TRAINING")
else:
    print("\n⚠️ Some checks failed. Do NOT start training yet.")

       FINAL PREPROCESSING & DATA SAVING CHECK

📁 Saved File Check
Path: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/processed_potato_data.npz
File exists: True
File size: 1.21 GB

📊 Dataset Shapes
X_train: (1721, 224, 224, 3)
y_train: (1721,)
X_val  : (215, 224, 224, 3)
y_val  : (215,)
X_test : (216, 224, 224, 3)
y_test : (216,)

📦 Total Images
Training   : 1721
Validation : 215
Testing    : 216
TOTAL      : 2152

🎨 Pixel Range
Minimum pixel value: 0.0
Maximum pixel value: 1.0

🌱 Class Distribution
Healthy         | Train:  121 | Val:   15 | Test:   16
Early_Blight    | Train:  800 | Val:  100 | Test:  100
Late_Blight     | Train:  800 | Val:  100 | Test:  100

FINAL CHECK RESULTS
✅ File exists
✅ Total images = 2152
✅ Image shape correct
✅ Train labels match
✅ Validation labels match
✅ Test labels match
✅ Normalization correct

🎉 PREPROCESSING + DATA SAVING = 100% VERIFIED
🚀 Dataset is READY for CNN MODEL TRAINING
